# Transformer data prep - gold

Turns gold's daily returns into the windowed examples a transformer needs: many overlapping `(30 days in -> next day's move out)` pairs, split into train/validation/test in time order, and packaged into PyTorch's data format.

The target for each example is the same realized-volatility proxy used in the GARCH backtest (next day's absolute return) - so the final RMSE comparison against GARCH's 0.9032 is apples-to-apples.

In [1]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader

WINDOW_SIZE = 30

In [2]:
gold = pd.read_csv("../data/gold_futures.csv", skiprows=[1, 2], index_col=0, parse_dates=True)
gold.index.name = "Date"

returns = gold["Close"].pct_change().dropna() * 100
len(returns)

4170

## Build the sliding windows

For each position, take the previous `WINDOW_SIZE` (30) returns as input, and the very next day's absolute return as the target. Sliding this one day at a time across the whole series produces many overlapping examples.

In [3]:
def create_windows(returns, window_size):
    values = returns.values
    X, y, target_dates = [], [], []
    for start in range(len(values) - window_size):
        end = start + window_size
        X.append(values[start:end])
        y.append(abs(values[end]))
        target_dates.append(returns.index[end])
    return (
        np.array(X, dtype=np.float32),
        np.array(y, dtype=np.float32),
        pd.DatetimeIndex(target_dates),
    )


X, y, target_dates = create_windows(returns, WINDOW_SIZE)
print(f"X shape: {X.shape}  (examples, days per window)")
print(f"y shape: {y.shape}  (one target per example)")

X shape: (4140, 30)  (examples, days per window)
y shape: (4140,)  (one target per example)


## Split into train / validation / test, in time order

70% train, 15% validation, 15% test - no shuffling across these boundaries. Validation is new compared to the GARCH backtest: since training runs for many repeated passes over the data, we need a way to check for overfitting *during* training, not just at the end.

In [4]:
n = len(X)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train, y_train = X[:train_end], y[:train_end]
X_val, y_val = X[train_end:val_end], y[train_end:val_end]
X_test, y_test = X[val_end:], y[val_end:]

print(f"Train: {len(X_train)} examples, {target_dates[0].date()} to {target_dates[train_end - 1].date()}")
print(f"Val:   {len(X_val)} examples, {target_dates[train_end].date()} to {target_dates[val_end - 1].date()}")
print(f"Test:  {len(X_test)} examples, {target_dates[val_end].date()} to {target_dates[-1].date()}")

Train: 2898 examples, 2010-02-18 to 2021-08-25
Val:   621 examples, 2021-08-26 to 2024-02-14
Test:  621 examples, 2024-02-15 to 2026-08-05


## Normalize the inputs

Neural networks train more reliably when inputs are roughly centered around 0 with a consistent scale. We standardize using the mean/std of the *training* data only - using test-set statistics here would leak information the model shouldn't have access to yet, the same leakage principle we were careful about with the GARCH backtest. The targets (`y`) are left in their original percent units, so the final RMSE stays directly comparable to GARCH's score.

In [5]:
train_mean = X_train.mean()
train_std = X_train.std()

X_train_norm = (X_train - train_mean) / train_std
X_val_norm = (X_val - train_mean) / train_std
X_test_norm = (X_test - train_mean) / train_std

print(f"Training mean: {train_mean:.4f}, training std: {train_std:.4f}")

Training mean: 0.0220, training std: 1.0290


## Package into PyTorch tensors and DataLoaders

Each window gets an extra dimension added (`.unsqueeze(-1)`) - PyTorch expects each day in the window to be a small *vector* of features, even though right now that vector only holds one number (the return). This leaves room to add more features (like realized volatility) later without changing the model's structure.

`DataLoader` feeds the model small batches at a time during training rather than everything at once. `shuffle=True` on the training loader mixes up *which* windows appear in which batch - this is different from, and doesn't violate, the "no shuffling across time" rule from the train/val/test split: each window's internal 30 days stays correctly ordered, we're just not always processing window #1 before window #500 during training.

In [6]:
def to_dataset(X, y):
    X_t = torch.tensor(X, dtype=torch.float32).unsqueeze(-1)
    y_t = torch.tensor(y, dtype=torch.float32)
    return TensorDataset(X_t, y_t)


train_dataset = to_dataset(X_train_norm, y_train)
val_dataset = to_dataset(X_val_norm, y_val)
test_dataset = to_dataset(X_test_norm, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

batch_X, batch_y = next(iter(train_loader))
print(f"One batch of X: {batch_X.shape}  (batch, window_size, features)")
print(f"One batch of y: {batch_y.shape}")

One batch of X: torch.Size([64, 30, 1])  (batch, window_size, features)
One batch of y: torch.Size([64])
